# TSB Triplet Visualization (Amazon, Favorita, M5)

Este notebook visualiza el comportamiento del modelo **TSB** en los 3 productos proxy inferidos entre datasets:
- Paper roll
- Soap/Cleaner
- Personal care


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repo root so imports from src/ work regardless of notebook cwd.
def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'src').exists() and (cur / 'reports').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.run_cross_market_peak_comparison import (
    _load_amazon_selected,
    _load_favorita_selected,
    _load_m5_selected,
)
from src.models.tsb_model import TSBModel

print('Repo root:', REPO_ROOT)


In [ ]:
triplets_csv = REPO_ROOT / 'reports' / 'cross_market_peak_comparison' / 'inferred_triplets.csv'
amazon_file = 'Health_and_Household.jsonl.gz'
amazon_max_rows = 100_000
test_days = 365

triplets = pd.read_csv(triplets_csv)
triplets


In [ ]:
m5_ids = triplets['M5_WALMART'].astype(str).tolist()
fav_ids = triplets['FAVORITA'].astype(str).tolist()
amz_ids = triplets['AMAZON_2023'].astype(str).tolist()

series_map_by_ds = {
    'M5_WALMART': _load_m5_selected(str(REPO_ROOT / 'data' / 'raw' / 'm5'), m5_ids),
    'FAVORITA': _load_favorita_selected(str(REPO_ROOT / 'data' / 'raw' / 'favorita'), fav_ids, store_nbr=1),
    'AMAZON_2023': _load_amazon_selected(
        str(REPO_ROOT / 'data' / 'raw' / 'amazon_2023' / 'review_categories'),
        amazon_file,
        amz_ids,
        max_rows=amazon_max_rows,
    ),
}

{k: len(v) for k, v in series_map_by_ds.items()}


In [ ]:
def split_ts(df: pd.DataFrame, test_days: int = 365):
    df = df.sort_values('date').reset_index(drop=True).copy()
    if len(df) <= test_days + 30:
        test_days = max(30, int(len(df) * 0.2))
    train = df.iloc[:-test_days].copy()
    test = df.iloc[-test_days:].copy()
    return train, test

def evaluate_tsb(df: pd.DataFrame, test_days: int = 365):
    train_df, test_df = split_ts(df, test_days=test_days)
    y_train = train_df['sales'].astype(float).values
    y_test = test_df['sales'].astype(float).values

    model = TSBModel().fit(y_train)
    y_pred, _, _ = model.forecast(len(y_test))

    mae = float(np.mean(np.abs(y_test - y_pred)))
    rmse = float(np.sqrt(np.mean((y_test - y_pred) ** 2)))
    mape_mask = y_test > 0
    mape = float(np.mean(np.abs((y_test[mape_mask] - y_pred[mape_mask]) / y_test[mape_mask])) * 100) if mape_mask.sum() > 0 else np.nan

    return {
        'train_df': train_df,
        'test_df': test_df,
        'y_test': y_test,
        'y_pred': y_pred,
        'mae': mae,
        'rmse': rmse,
        'mape': mape,
        'alpha': model.alpha,
        'beta': model.beta,
    }


In [ ]:
rows = []
pred_store = {}
datasets = ['AMAZON_2023', 'FAVORITA', 'M5_WALMART']

for _, trip in triplets.iterrows():
    inferred_product = trip['inferred_product']
    for ds in datasets:
        sid = str(trip[ds])
        df = series_map_by_ds[ds][sid].copy()
        out = evaluate_tsb(df, test_days=test_days)

        rows.append({
            'inferred_product': inferred_product,
            'dataset': ds,
            'series_id': sid,
            'mae': out['mae'],
            'rmse': out['rmse'],
            'mape': out['mape'],
            'tsb_alpha': out['alpha'],
            'tsb_beta': out['beta'],
            'train_days': len(out['train_df']),
            'test_days': len(out['test_df']),
        })

        pred_store[(inferred_product, ds)] = out

results = pd.DataFrame(rows).sort_values(['inferred_product', 'dataset']).reset_index(drop=True)
results


In [ ]:
pivot_mae = results.pivot(index='inferred_product', columns='dataset', values='mae')
pivot_rmse = results.pivot(index='inferred_product', columns='dataset', values='rmse')
pivot_mape = results.pivot(index='inferred_product', columns='dataset', values='mape')

display(pivot_mae.round(4))
display(pivot_rmse.round(4))
display(pivot_mape.round(2))


In [ ]:
dataset_colors = {
    'AMAZON_2023': '#d62728',
    'FAVORITA': '#2ca02c',
    'M5_WALMART': '#1f77b4',
}

products = triplets['inferred_product'].tolist()
datasets = ['AMAZON_2023', 'FAVORITA', 'M5_WALMART']

fig, axes = plt.subplots(nrows=len(products), ncols=len(datasets), figsize=(18, 11), sharex=False)

for i, product in enumerate(products):
    for j, ds in enumerate(datasets):
        ax = axes[i, j] if len(products) > 1 else axes[j]
        out = pred_store[(product, ds)]

        test_df = out['test_df']
        y_test = out['y_test']
        y_pred = out['y_pred']

        ax.plot(test_df['date'], y_test, color='black', lw=1.2, label='Actual')
        ax.plot(test_df['date'], y_pred, color=dataset_colors[ds], lw=1.6, label='TSB')

        ax.set_title(f"{product} | {ds}", fontsize=10)
        ax.tick_params(axis='x', rotation=35)
        ax.grid(alpha=0.2)

        if i == 0 and j == 0:
            ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Optional: save current metric table for report integration
out_dir = REPO_ROOT / 'reports' / 'tsb_triplet_experiment'
out_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(out_dir / 'tsb_triplet_summary_from_notebook.csv', index=False)
results
